# Kou jump-diffusion: asymmetric double-exponential jumps

Merton's jumps are normal in log-space. That has two limitations: a normal is
symmetric in shape (up and down jumps differ only in mean, not form), and it has
thin Gaussian tails. Real markets crash more sharply than they rally, and their
extreme moves are fatter-tailed than a Gaussian jump captures.

Kou (2002) replaces the normal jump with an asymmetric double-exponential. This
notebook derives the Kou characteristic function, then implements and validates
it, reusing the model-agnostic COS core and the Monte Carlo validation pattern
built for Merton. The only genuinely new derivation is the single-jump
characteristic function; pricing and validation are reuse.

## 1. The model

Same diffusion-plus-compound-Poisson structure as Merton, differing only in the
jump size distribution:

$$d(\log S_t) = \left(r - q - \tfrac12\sigma^2 - \lambda\kappa\right)dt
+ \sigma\, dW_t + dJ_t$$

The jump size $Y$ (in log-space) follows an **asymmetric double-exponential**:

$$f_Y(y) = p\,\eta_1 e^{-\eta_1 y}\mathbf{1}_{y \ge 0}
+ (1-p)\,\eta_2 e^{\eta_2 y}\mathbf{1}_{y < 0}$$

Two one-sided exponentials glued at zero:

- With probability $p$: an **up-jump**, $\text{Exp}(\eta_1)$, mean size $1/\eta_1$.
- With probability $1-p$: a **down-jump**, $\text{Exp}(\eta_2)$, mean size $1/\eta_2$.

Jump parameters: intensity $\lambda$ (jumps/year), up-probability $p$, up-rate
$\eta_1$, down-rate $\eta_2$. Four jump parameters vs Merton's two.

What the extra parameters buy:

- **Asymmetry of shape.** $\eta_1 \ne \eta_2$ makes up- and down-jumps genuinely
  different in form, not just mean. Typically $\eta_2 < \eta_1$ (down-jumps decay
  slower, so fatter) and $p < 0.5$ (down-jumps more likely): structural crash
  asymmetry.
- **Fatter tails.** Exponential tails are heavier than Gaussian, so extreme
  jumps are more probable, matching the deep-wing richness of real option prices.
- **Exotic tractability.** The double-exponential's memorylessness makes barrier
  and lookback options analytically tractable under Kou where they are not under
  Merton. Not used in this European calibration, but it is why Kou is prized.

### The constraint $\eta_1 > 1$

The up-rate must exceed 1. The reason surfaces in the compensator below: it
involves $E[e^Y]$, and for an up-jump $e^Y$ with $Y \sim \text{Exp}(\eta_1)$ that
expectation only converges if $\eta_1 > 1$. Economically, up-jumps cannot be so
fat-tailed that the expected price multiplier diverges. It is the Kou analogue of
"the jump has finite mean," and violating it makes $\kappa$ blow up.

## 2. No-arbitrage and the compensator (the FTAP thread)

The compensator $-\lambda\kappa$ in the drift is not cosmetic; it is what makes
the model arbitrage-free. The chain of reasoning ties back to the Fundamental
Theorem of Asset Pricing.

FTAP: no arbitrage holds iff there is an equivalent measure under which the
*discounted* price is a martingale. Writing the model under the risk-neutral
measure $\mathbb{Q}$ asserts we are in that measure, so we require

$$E^{\mathbb{Q}}[S_T] = S_0 e^{(r-q)T}$$

Jumps add expected growth: each jump multiplies the price by $e^Y$, and
$E[e^Y] \ne 1$ in general. Left uncorrected, this pushes $E[S_T]$ above
$S_0 e^{(r-q)T}$, breaking the martingale property and admitting arbitrage. The
compensator removes exactly that excess growth:

$$\kappa = E[e^Y - 1] = E[e^Y] - 1$$

is the expected proportional price change per jump, $\lambda\kappa$ is the
expected jump growth rate, and subtracting it in the drift restores the
martingale.

### $\phi(-i)$ computes it, and checks it

The characteristic function is $\phi(u) = E[e^{iu \log S_T}]$. At $u = -i$,
$e^{iu \log S_T} = e^{\log S_T} = S_T$, so $\phi(-i) = E[S_T]$, which must equal
$S_0 e^{(r-q)T}$. The same trick at the jump level: the single-jump char func is
$\hat f(u) = E[e^{iuY}]$, so $\hat f(-i) = E[e^Y]$, giving

$$\kappa = \hat f(-i) - 1$$

So $u = -i$ turns the char func's $e^{iuY}$ into the price multiplier $e^Y$. This
is both how the compensator is computed and how the martingale property is
verified (evaluate $\phi(-i)$, confirm it equals $S_0 e^{(r-q)T}$), which becomes
a validation gate.

## 3. The characteristic function, derived

Same Levy-Khintchine structure as Merton, since the compound-Poisson form is
identical and only the jump size distribution changed:

$$\phi_{\text{Kou}}(u) = \phi_{\text{diffusion}}(u) \cdot
\exp\!\Big(\lambda\tau\big[\hat f(u) - 1\big]\Big)$$

The diffusion factor is the known BSM Gaussian (with jump-compensated drift). The
only new work is $\hat f(u)$, the char func of a single double-exponential jump.

### The single-jump char func

By definition $\hat f(u) = E[e^{iuY}] = \int_{-\infty}^{\infty} e^{iuy} f_Y(y)\,dy$.
Split at zero into the up and down pieces. Each is a standard one-sided
exponential integral:

$$\int_0^\infty \eta e^{-\eta y} e^{iuy}\,dy = \frac{\eta}{\eta - iu}$$

Applying it to each side (the down-jump lives on $y < 0$, flipping the sign on
$iu$), weighted by the up/down probabilities:

$$\hat f(u) = \frac{p\,\eta_1}{\eta_1 - iu} + \frac{(1-p)\,\eta_2}{\eta_2 + iu}$$

The two denominators carry opposite signs on $iu$ because the up-jump is
supported on $[0,\infty)$ and the down-jump on $(-\infty, 0)$.

### The compensator

From $\kappa = \hat f(-i) - 1$, substitute $u = -i$ (so $iu = 1$ in the first
denominator, $iu = 1$ flips to $-1$... careful with signs):

$$\kappa = \frac{p\,\eta_1}{\eta_1 - 1} + \frac{(1-p)\,\eta_2}{\eta_2 + 1} - 1$$

The $\eta_1 - 1$ in the first denominator is exactly where the $\eta_1 > 1$
constraint bites: at $\eta_1 = 1$ it is a division by zero, and below it the term
goes negative, an unphysical expected up-jump multiplier. (No constraint on
$\eta_2$ beyond $\eta_2 > 0$, since $\eta_2 + 1$ is always safe; down-jumps shrink
the price by a bounded factor.)

### The full Kou char func

$$\phi_{\text{Kou}}(u) = \exp\!\Big(iu\big(\log S_0 + (r - q - \tfrac12\sigma^2
- \lambda\kappa)\tau\big) - \tfrac12\sigma^2 u^2\tau\Big)
\cdot \exp\!\Big(\lambda\tau\big[\hat f(u) - 1\big]\Big)$$

Same sanity checks as Merton, each a validation gate: $\lambda \to 0$ recovers
BSM; $\phi(-i) = S_0 e^{(r-q)T}$ certifies $\kappa$; $\phi(0) = 1$.

## 4. The cumulants

Diffusion and jump cumulants add (independence), as for Merton. The diffusion
cumulants are BSM's; the jump cumulants come from the compound-Poisson moments
of the double-exponential.

The single-jump moments (mean and second moment of $Y$):

$$E[Y] = \frac{p}{\eta_1} - \frac{1-p}{\eta_2}, \qquad
E[Y^2] = \frac{2p}{\eta_1^2} + \frac{2(1-p)}{\eta_2^2}$$

($E[Y]$ is the probability-weighted up-mean minus down-mean; $E[Y^2]$ uses that
$\text{Exp}(\eta)$ has second moment $2/\eta^2$.) Then, exactly as for Merton:

$$c_1 = \log S_0 + \left(r - q - \tfrac12\sigma^2 - \lambda\kappa\right)\tau
+ \lambda\tau\, E[Y]$$

$$c_2 = \sigma^2\tau + \lambda\tau\, E[Y^2]$$

The jump variance is $\lambda\tau\, E[Y^2]$, the compound-Poisson second moment,
same structure as Merton's $\lambda\tau(\mu_J^2 + \delta_J^2)$ but with the
double-exponential moments. Getting $E[Y^2]$ right (both the $2/\eta^2$ factors)
keeps the truncation range and thus the wings accurate.

In [ ]:
# models/kou.py will hold the char func + cumulants; pricing wrapper in fourier.py.
#
# def kou_char_func(u, S0, r, q, T, sigma, lam, p, eta1, eta2):
#     """Kou characteristic function of log(S_T).
#     Requires eta1 > 1 for the compensator to converge.
#     f_hat(u) = p*eta1/(eta1 - iu) + (1-p)*eta2/(eta2 + iu)
#     kappa    = p*eta1/(eta1 - 1) + (1-p)*eta2/(eta2 + 1) - 1
#     phi = diffusion_factor * exp(lam*T*(f_hat(u) - 1))
#     """
#     raise NotImplementedError
#
# def kou_cumulants(S0, r, q, T, sigma, lam, p, eta1, eta2):
#     """c1, c2 with E[Y] = p/eta1 - (1-p)/eta2, E[Y^2] = 2p/eta1^2 + 2(1-p)/eta2^2."""
#     raise NotImplementedError
#
# def kou_simulate_terminal(...):
#     """One-step MC. Per jump: Bernoulli(p) up/down, then Exp(eta1) or Exp(eta2).
#     Vectorized over the per-path total number of jumps. Same kappa as the char func."""
#     raise NotImplementedError

In [ ]:
# Validation, mirroring Merton:
#   Gate 1: lam=0 recovers BSM (call + put).
#   Gate 2: phi(-i) = S0 exp((r-q)T), certifies kappa.
#   Gate 3: phi(0) = 1.
#   Gate 4: cumulants reduce to BSM at lam=0.
#   Gate 5: COS vs MC (z-score < ~3), the real jump-pricing test.
#   Gate 6: eta1 <= 1 should raise or be rejected (the constraint).